In [1]:
import pandas as pd
import glob
import os

In [2]:

# 1. LOAD DIAMOND RESULTS
cols = [
    "query",
    "target",
    "identity",
    "aln_len",
    "mismatch",
    "gap",
    "qstart",
    "qend",
    "tstart",
    "tend",
    "evalue",
    "bitscore"
]

hits = pd.read_csv("unique_sq_hits_all.tsv", sep="\t", names=cols)

hits.head()

,query,target,identity,aln_len,mismatch,gap,qstart,qend,tstart,tend,evalue,bitscore
0,AAK90112.1,GCA_902803065.1_CCEEBC_01630,28.9,201,127,5,24,218,42,232,3.600000e-09,55.5
1,AAK90112.1,GCA_902803065.1_CCEEBC_01592,27.9,201,129,4,24,218,42,232,4.830000e-09,55.1
2,AAK90112.1,GCA_902803055.1_MMPJKM_00615,27.5,182,95,7,1,167,1,160,4.970000e-06,45.8
3,WP_017967307.1,GCA_902803065.1_CCEEBC_00013,35.1,544,334,9,29,563,15,548,2.750000e-100,313.0
4,WP_017967307.1,GCA_902803035.1_BOCBGN_01878,35.0,534,329,10,41,566,26,549,4.870000e-92,292.0


In [3]:
mask = hits['query'].str.contains(r'\|', na=False)
hits.loc[mask, 'query'] = hits.loc[mask, 'query'].str.split(r'\|').str[1]

In [4]:
# 2. PARSE TARGET IDS

hits["MAG"] = hits["target"].str.extract(r"((?:GCA|GCF)_\d+\.\d+)")
hits["locus_tag"] = hits["target"].str.split("_").str[-2:].str.join("_")

In [ ]:
# 3. PARSE GFF FILES

gff_records = []

gff_files = glob.glob("/beegfs/lvsea/bakta_annotation/*/*/*.gff3")

for gff in gff_files:

    mag = os.path.basename(gff).replace(".gff3","")

    with open(gff) as f:

        for line in f:

            if line.startswith("#"):
                continue

            parts = line.strip().split("\t")

            if len(parts) < 9 or parts[2] != "CDS":
                continue

            contig = parts[0]
            start = parts[3]
            end = parts[4]
            attrs = parts[8]

            attr_dict = {}
            for item in attrs.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    attr_dict[k] = v

            locus = attr_dict.get("locus_tag")
            product = attr_dict.get("product")

            gff_records.append({
                "MAG": mag,
                "locus_tag": locus,
                "contig": contig,
                "start": start,
                "end": end,
                "product": product
            })

gff_df = pd.DataFrame(gff_records)

In [ ]:
# 4. MERGE TABLES

meta = hits.merge(gff_df, on=["MAG", "locus_tag"], how="left")

KeyError: 'MAG'

In [ ]:
# 5. ADD TAXONOMY

# читаем таблицу с таксономией
tax = pd.read_csv("gtdbtk_summary_all.csv", sep="\t")

# приводим MAG к одному формату
tax["MAG"] = tax["user_genome"].str.replace("MAG_", "", regex=False)
tax = tax.rename(columns={"classification": "taxonomy"})
tax = tax[["MAG", "taxonomy"]]

# чистим пробелы
meta["MAG"] = meta["MAG"].astype(str).str.strip()
tax["MAG"] = tax["MAG"].astype(str).str.strip()

# merge с таксономией
meta = meta.merge(tax, on="MAG", how="left", validate="many_to_one")

In [ ]:
# 6. SELECT FINAL COLUMNS

meta = meta[[
    "MAG",
    "query",
    "target",
    "identity",
    "aln_len",
    "evalue",
    "bitscore",
    "product",
    "contig",
    "start",
    "end"
]]

In [ ]:
# 7. SAVE

meta.to_csv("SQ_metatable_all_parts_V2.tsv", sep="\t", index=False)